# Fine-Tuning TrOCR untuk Handwritten Kwitansi (Google Colab)

Notebook ini fine-tune model **microsoft/trocr-base-handwritten** pada dataset tulisan tangan kwitansi Indonesia.

**Format dataset CSV** (2 kolom):

```
image_path,text
data/imgs/kwitansi_001_line_01.png,Seratus Ribu Rupiah
data/imgs/kwitansi_001_line_02.png,Kwitansi No. 001/ABC/2026
```

> Gambar harus berupa **crop per baris** (tinggi ±64–384px).
> Jika dataset masih full-page, gunakan cell segmentasi baris di bawah.

Cara pakai: **Runtime > Change runtime type > GPU (T4)**, lalu jalankan semua cell dari atas ke bawah.

In [ ]:
# @title 1. Install dependencies (~2 menit)
%pip install -q transformers datasets evaluate jiwer accelerate peft pillow opencv-python-headless

In [ ]:
# @title 2. Konfigurasi
import os
import torch

# --- Path dataset CSV (kolom: image_path,text) ---
# Path relatif akan dicari di dalam DATASET_ROOT (lihat cell 3b).
TRAIN_CSV = "data/train_combined.csv"   # @param {type:"string"}
# --- Base model HuggingFace ---
MODEL_NAME = "microsoft/trocr-base-handwritten"   # @param ["microsoft/trocr-base-handwritten", "microsoft/trocr-large-handwritten"]
# --- Output folder model hasil training ---
OUTPUT_DIR = "models/trocr-kwitansi"      # @param {type:"string"}

# --- Hyperparameters ---
EPOCHS = 50                # @param {type:"integer"}
BATCH_SIZE = 8             # @param {type:"integer"}
LEARNING_RATE = 5e-5       # @param {type:"number"}
MAX_LENGTH = 128           # @param {type:"integer"}
VAL_SPLIT = 0.1            # @param {type:"number"}
MAX_GRAD_NORM = 1.0

# --- LoRA (hemat memori ~70%, fine-tune hanya ~0.1% parameter) ---
USE_LORA = True            # @param {type:"boolean"}
LORA_R = 8                 # @param {type:"integer"}
LORA_ALPHA = 32            # @param {type:"integer"}

GRADIENT_CHECKPOINTING = True   # @param {type:"boolean"}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Device: {DEVICE}")
if DEVICE == "cpu":
    print("[WARN] GPU tidak aktif! Runtime > Change runtime type > GPU")

## Pilih sumber dataset

- **Opsi A**: Upload file ZIP berisi gambar + CSV
- **Opsi B**: Mount Google Drive dan arahkan `TRAIN_CSV` ke file CSV di Drive
- **Opsi C**: Belum punya dataset? Gunakan generator dataset sintetis (untuk tes pipeline)

In [ ]:
# @title 3a. Opsi A — Upload ZIP dataset {display-mode:"form"}
# ZIP berisi folder gambar + file CSV. Contoh struktur:
#   dataset.zip
#     ├── imgs/  (gambar crop per baris)
#     └── train.csv  (kolom image_path,text)
USE_UPLOAD = False   # @param {type:"boolean"}

if USE_UPLOAD:
    import zipfile
    import google.colab.files as colab_files
    print("Upload file ZIP dataset Anda:")
    uploaded = colab_files.upload()
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, "r") as zf:
        zf.extractall("/content/")
    print("[INFO] Dataset diekstrak ke /content/")
else:
    print("[SKIP] Opsi upload tidak dipakai.")

In [ ]:
# @title 3b. Opsi B — Google Drive {display-mode:"form"}
# Mount Drive lalu set TRAIN_CSV ke path CSV di Drive.
# Contoh: /content/drive/MyDrive/dataset/train.csv
# Pastikan kolom image_path di CSV adalah path ABSOLUT atau relatif terhadap
# root project (/content). Gunakan USE_DRIVE untuk memperbaiki path otomatis.
USE_DRIVE = False   # @param {type:"boolean"}
DATASET_ROOT_IN_DRIVE = "/content/drive/MyDrive/FineTune-TrOCR"   # @param {type:"string"}

if USE_DRIVE:
    import google.colab.drive as drive_mod
    drive_mod.mount("/content/drive")
    DATASET_ROOT = DATASET_ROOT_IN_DRIVE
    print(f"[INFO] Dataset root: {DATASET_ROOT}")
else:
    DATASET_ROOT = "/content"
    print("[SKIP] Google Drive tidak dipakai.")

In [ ]:
# @title 3c. Opsi C — Generator dataset sintetis (tes pipeline) {display-mode:"form"}
# Membuat contoh dataset sintetis (teks cetak pada gambar) supaya pipeline
# bisa dites end-to-end tanpa data asli. TIDAK perlu dijalankan jika sudah
# punya dataset sendiri lewat Opsi A/B.
USE_SYNTHETIC = True   # @param {type:"boolean"}
N_SYNTHETIC = 200      # @param {type:"integer"}

if USE_SYNTHETIC:
    import csv as csv_mod
    import random as random_mod
    import pathlib as pathlib_mod

    import cv2 as cv2_mod
    import numpy as np_mod

    synth_dir = pathlib_mod.Path("/content/synthetic_dataset")
    imgs_dir = synth_dir / "imgs"
    imgs_dir.mkdir(parents=True, exist_ok=True)

    words = [
        "Kwitansi", "Nomor", "Tanggal", "Terima", "dari", "sebanyak",
        "Seratus", "Ribu", "Rupiah", "Lima", "Puluh", "Untuk",
        "pembayaran", "Jumlah", "PT", "Maju", "Jaya", "Jakarta",
        "2026", "001", "ABC", "Invoice", "Barang", "Sesuai",
    ]
    fonts_to_try = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
    ]
    font_path = None
    for fp in fonts_to_try:
        if pathlib_mod.Path(fp).is_file():
            font_path = fp
            break
    if font_path is None:
        # fallback: pakai font bawaan OpenCV (HERSHEY_SIMPLEX)
        font_path = None

    rows = []
    rng = random_mod.Random(42)
    for i in range(N_SYNTHETIC):
        n_words = rng.randint(3, 7)
        text = " ".join(rng.choice(words) for _ in range(n_words))
        img = np_mod.full((64, 800, 3), 255, dtype=np_mod.uint8)
        if font_path is not None:
            pil_ok = False
        # gunakan cv2 putText agar tidak butuh PIL font handling
        scale = rng.uniform(1.1, 1.6)
        thickness = rng.randint(1, 2)
        y_pos = 42
        x_pos = rng.randint(5, 30)
        cv2_mod.putText(img, text, (x_pos, y_pos),
                        cv2_mod.FONT_HERSHEY_SIMPLEX, scale,
                        (rng.randint(0, 60), rng.randint(0, 60), rng.randint(0, 60)),
                        thickness, cv2_mod.LINE_AA)
        # noise ringan agar mirip hasil scan
        noise = np_mod.random.normal(0, 6, img.shape).astype(np_mod.uint8)
        img = np_mod.clip(img.astype(int) + noise, 0, 255).astype(np_mod.uint8)
        img_path = imgs_dir / f"synth_{i:04d}.png"
        cv2_mod.imwrite(str(img_path), img)
        rows.append({"image_path": str(img_path), "text": text})

    TRAIN_CSV = str(synth_dir / "train.csv")
    with open(TRAIN_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv_mod.DictWriter(f, fieldnames=["image_path", "text"])
        writer.writeheader()
        writer.writerows(rows)
    print(f"[INFO] Dataset sintetis: {len(rows)} baris -> {TRAIN_CSV}")
else:
    print("[SKIP] Dataset sintetis tidak dibuat.")

In [ ]:
# @title 3d. (Opsional) Segmentasi gambar full-page menjadi baris {display-mode:"form"}
# Jalankan ini HANYA jika dataset Anda masih full-page kwitansi.
# Hasilnya: crop per baris + CSV template yang kolom 'text'-nya harus
# Anda isi manual sebelum training.
RUN_SEGMENTATION = False          # @param {type:"boolean"}
SEG_IMAGES_DIR = "/content/kwitansi_pages"    # @param {type:"string"}
SEG_OUTPUT_DIR = "/content/data/segmented_lines"  # @param {type:"string"}
SEG_TARGET_HEIGHT = 384
SEG_MAX_WIDTH = 1024

if RUN_SEGMENTATION:
    import csv as seg_csv
    import pathlib as seg_pathlib
    import cv2 as seg_cv2

    os.makedirs(SEG_OUTPUT_DIR, exist_ok=True)
    out_path = seg_pathlib.Path(SEG_OUTPUT_DIR)
    seg_rows = []
    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

    for img_file in sorted(seg_pathlib.Path(SEG_IMAGES_DIR).rglob("*")):
        if not img_file.is_file() or img_file.suffix.lower() not in image_exts:
            continue
        print(f"  Segmentasi: {img_file.name}")
        img = seg_cv2.imread(str(img_file))
        if img is None:
            continue
        gray = seg_cv2.cvtColor(img, seg_cv2.COLOR_BGR2GRAY)
        binary = seg_cv2.adaptiveThreshold(
            ~gray, 255, seg_cv2.ADAPTIVE_THRESH_MEAN_C, seg_cv2.THRESH_BINARY, 15, -2
        )
        h, w = img.shape[:2]
        kernel_w = max(10, w // 40)
        kernel_h = max(3, 15 // 2)
        kernel = seg_cv2.getStructuringElement(seg_cv2.MORPH_RECT, (kernel_w, kernel_h))
        dilated = seg_cv2.dilate(binary, kernel, iterations=1)

        profile = dilated.sum(axis=1) / 255.0
        threshold = max(1.0, float(profile.max()) * 0.02)
        ink_rows = profile > threshold

        stem = img_file.stem
        line_idx = 0
        in_line = False
        start = 0
        ink_list = list(ink_rows) + [False]
        for y in range(len(ink_list)):
            has_ink = bool(ink_list[y])
            if has_ink and not in_line:
                in_line = True
                start = y
            elif not has_ink and in_line:
                in_line = False
                end = y
                if end - start >= 15:
                    pad = 4
                    y0 = max(0, start - pad)
                    y1 = min(h, end + pad)
                    crop = img[y0:y1, :]
                    crop_h, crop_w = crop.shape[:2]
                    scale = SEG_TARGET_HEIGHT / float(crop_h)
                    new_w = min(int(crop_w * scale), SEG_MAX_WIDTH)
                    interp = seg_cv2.INTER_CUBIC if scale > 1 else seg_cv2.INTER_AREA
                    resized = seg_cv2.resize(crop, (max(1, new_w), SEG_TARGET_HEIGHT), interpolation=interp)
                    crop_name = f"{stem}_line_{line_idx:03d}.png"
                    crop_path = out_path / crop_name
                    seg_cv2.imwrite(str(crop_path), resized)
                    seg_rows.append({"image_path": str(crop_path), "text": ""})
                    line_idx += 1

    csv_file = out_path / "dataset.csv"
    with open(csv_file, "w", newline="", encoding="utf-8") as f:
        writer = seg_csv.DictWriter(f, fieldnames=["image_path", "text"])
        writer.writeheader()
        writer.writerows(seg_rows)
    print(f"\n[INFO] Total baris tersegmentasi: {len(seg_rows)}")
    print(f"[INFO] Isi kolom 'text' di {csv_file} secara manual, lalu set:")
    print(f"       TRAIN_CSV = \"{csv_file}\"")
else:
    print("[SKIP] Segmentasi dilewati.")

In [ ]:
# @title 4. Load dataset dari CSV
import csv as ds_csv
import random as ds_random
import pathlib as ds_pathlib

def load_dataset(csv_path, val_split=0.1):
    rows = []
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = ds_csv.DictReader(f)
        for row in reader:
            img_path = row["image_path"].strip()
            text = row["text"].strip()
            if not ds_pathlib.Path(img_path).is_absolute():
                img_path = str(ds_pathlib.Path(DATASET_ROOT) / img_path)
            if text and ds_pathlib.Path(img_path).is_file():
                rows.append({"image_path": img_path, "text": text})

    if not rows:
        raise ValueError(
            f"Tidak ada baris valid di {csv_path}. Pastikan:\n"
            f"  1. Kolom 'text' tidak kosong\n"
            f"  2. Path gambar valid (absolute atau relatif thd {DATASET_ROOT})"
        )

    ds_random.shuffle(rows)
    split = max(1, int(len(rows) * val_split))
    val_data = rows[:split]
    train_data = rows[split:] if len(rows) > split else rows
    return train_data, val_data

train_data, val_data = load_dataset(TRAIN_CSV, VAL_SPLIT)
print(f"[INFO] Train: {len(train_data)} baris | Val: {len(val_data)} baris")
print(f"Contoh: {train_data[0]}")

In [ ]:
# @title 5. Load processor, tokenizer, dan model
import transformers

print(f"[INFO] transformers version: {transformers.__version__}")

image_processor = transformers.ViTImageProcessor.from_pretrained(MODEL_NAME)
tokenizer = transformers.RobertaTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = transformers.VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

# Setup decoder tokens
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.eos_token_id = tokenizer.sep_token_id

# generation_config (transformers >= 4.40: jangan set di model.config)
model.generation_config = transformers.GenerationConfig(
    decoder_start_token_id=tokenizer.cls_token_id,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.sep_token_id,
    max_length=MAX_LENGTH,
    early_stopping=True,
    no_repeat_ngram_size=3,
    length_penalty=2.0,
    num_beams=4,
)
print("[INFO] Model loaded.")

In [ ]:
# @title 6. Setup LoRA + gradient checkpointing (hemat memori)
import peft

if USE_LORA:
    print("[INFO] Menggunakan LoRA untuk memory-efficient fine-tuning")
    lora_config = peft.LoraConfig(
        task_type=peft.TaskType.SEQ_2_SEQ_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.1,
        target_modules=["q_proj", "v_proj"],
        bias="none",
    )
    model = peft.get_peft_model(model, lora_config)
    model.print_trainable_parameters()

if GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()
    if hasattr(model, "config"):
        model.config.use_cache = False

model.to(DEVICE)

In [ ]:
# @title 7. Dataset PyTorch + DataLoader
import torch.utils.data as torch_data
import PIL.Image as PilImage

class KwitansiDataset(torch_data.Dataset):
    def __init__(self, data, img_processor, tok, max_length=128):
        self.data = data
        self.image_processor = img_processor
        self.tokenizer = tok
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img_path = item["image_path"]
        if not ds_pathlib.Path(img_path).is_absolute():
            img_path = str(ds_pathlib.Path(DATASET_ROOT) / img_path)
        image = PilImage.open(img_path).convert("RGB")
        pixel_values = self.image_processor(images=image, return_tensors="pt").pixel_values.squeeze(0)
        labels = self.tokenizer(
            item["text"],
            return_tensors="pt",
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
        ).input_ids.squeeze(0)
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values, "labels": labels}

train_ds = KwitansiDataset(train_data, image_processor, tokenizer, max_length=MAX_LENGTH)
val_ds = KwitansiDataset(val_data, image_processor, tokenizer, max_length=MAX_LENGTH)

train_loader = torch_data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = torch_data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"[INFO] Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
# @title 8. Optimizer & scheduler
if USE_LORA:
    trainable_params = [p for p in model.parameters() if p.requires_grad]
else:
    trainable_params = list(model.parameters())

optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

n_params = sum(p.numel() for p in trainable_params)
print(f"[INFO] Trainable params: {n_params:,}")
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# @title 9. Training loop {display-mode:"form"}
import json as train_json
import time as time_mod
import pathlib as train_pathlib

output_dir = train_pathlib.Path(OUTPUT_DIR)
best_val_loss = float("inf")
history = []

print(f"[INFO] Mulai training ({EPOCHS} epochs, batch_size={BATCH_SIZE}, lr={LEARNING_RATE})")
print(f"[INFO] Output: {output_dir}\n")

for epoch in range(1, EPOCHS + 1):
    t_start = time_mod.time()

    # ------------------- TRAIN -------------------
    model.train()
    train_loss = 0.0
    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch["pixel_values"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        optimizer.zero_grad()
        train_loss += loss.item()
        if (batch_idx + 1) % 10 == 0:
            print(f"  Epoch {epoch}/{EPOCHS} | Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_train_loss = train_loss / max(len(train_loader), 1)
    scheduler.step()

    # ------------------- VALIDATE -------------------
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for batch in val_loader:
            pixel_values = batch["pixel_values"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            outputs = model(pixel_values=pixel_values, labels=labels)
            val_loss += outputs.loss.item()

            # PeftModel butuh base_model untuk generate
            gen_model = model.base_model if hasattr(model, "base_model") else model
            generated = gen_model.generate(pixel_values, max_new_tokens=MAX_LENGTH, num_beams=4)
            preds = tokenizer.batch_decode(generated, skip_special_tokens=True)
            for i in range(len(preds)):
                gt_labels = labels[i]
                gt_tokens = [t for t in gt_labels if t != -100 and t != tokenizer.pad_token_id]
                gt_text = tokenizer.decode(gt_tokens, skip_special_tokens=True)
                if preds[i].strip().lower() == gt_text.strip().lower():
                    val_correct += 1
                val_total += 1

    avg_val_loss = val_loss / max(len(val_loader), 1)
    accuracy = val_correct / max(val_total, 1)
    lr_now = optimizer.param_groups[0]["lr"]
    elapsed = time_mod.time() - t_start

    print(
        f"Epoch {epoch}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | Accuracy: {accuracy:.2%} | "
        f"LR: {lr_now:.2e} | {elapsed:.1f}s"
    )

    history.append({
        "epoch": epoch,
        "train_loss": round(avg_train_loss, 4),
        "val_loss": round(avg_val_loss, 4),
        "accuracy": round(accuracy, 4),
        "lr": lr_now,
    })

    # ------------------- SAVE BEST -------------------
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        print(f"  -> New best! Saving to {output_dir}")
        if USE_LORA:
            lora_state = {k: v for k, v in model.state_dict().items() if "lora_" in k}
            torch.save(lora_state, str(output_dir / "lora_adapter.bin"))
            lora_info = {
                "r": LORA_R,
                "alpha": LORA_ALPHA,
                "target_modules": ["q_proj", "v_proj"],
            }
            with open(output_dir / "lora_config.json", "w") as f:
                train_json.dump(lora_info, f, indent=2)
        else:
            model.save_pretrained(str(output_dir))
        image_processor.save_pretrained(str(output_dir))
        tokenizer.save_pretrained(str(output_dir))

with open(output_dir / "training_history.json", "w", encoding="utf-8") as f:
    train_json.dump(history, f, indent=2, ensure_ascii=False)

print(f"\n[DONE] Training selesai. Best val loss: {best_val_loss:.4f}")
print(f"[DONE] Model terbaik tersimpan di: {output_dir}")

In [ ]:
# @title 10. Plot riwayat training
import matplotlib.pyplot as plt_mod

epochs_hist = [h["epoch"] for h in history]
fig, axes = plt_mod.subplots(1, 2, figsize=(14, 4))

axes[0].plot(epochs_hist, [h["train_loss"] for h in history], label="Train Loss")
axes[0].plot(epochs_hist, [h["val_loss"] for h in history], label="Val Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_hist, [h["accuracy"] for h in history], label="Accuracy", color="green")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Validation Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt_mod.tight_layout()
plt_mod.show()

## Simpan hasil ke Google Drive (opsional)
Jalankan cell ini jika ingin menyimpan model hasil training ke Drive.

In [ ]:
# @title 11. Copy model ke Google Drive {display-mode:"form"}
SAVE_TO_DRIVE = True   # @param {type:"boolean"}
DRIVE_SAVE_DIR = "/content/drive/MyDrive/trocr-kwitansi"   # @param {type:"string"}

if SAVE_TO_DRIVE:
    import shutil as shutil_mod
    import google.colab.drive as gdrive_mod
    gdrive_mod.mount("/content/drive")
    os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
    shutil_mod.copytree(OUTPUT_DIR, DRIVE_SAVE_DIR, dirs_exist_ok=True)
    print(f"[INFO] Model disalin ke: {DRIVE_SAVE_DIR}")
else:
    print("[SKIP]")

In [ ]:
# @title 12. Uji inference dengan model hasil fine-tuning
# Upload satu gambar crop baris untuk di-OCR.
import google.colab.files as infer_files
import PIL.Image as InferImage
import transformers as inf_transformers

print("Upload gambar baris kwitansi (crop per baris):")
uploaded_infer = infer_files.upload()
img_name = list(uploaded_infer.keys())[0]

test_image = InferImage.open(img_name).convert("RGB")

pixel_values_test = image_processor(images=test_image, return_tensors="pt").pixel_values
pixel_values_test = pixel_values_test.to(DEVICE)

gen_model_final = model.base_model if hasattr(model, "base_model") else model
gen_model_final.eval()
with torch.no_grad():
    generated_ids = gen_model_final.generate(pixel_values_test, max_new_tokens=MAX_LENGTH, num_beams=4)
result_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print(f"\nHasil OCR: {result_text}")

# tampilkan gambar
fig, ax = plt_mod.subplots(figsize=(12, 3))
ax.imshow(test_image)
ax.axis("off")
ax.set_title(f"OCR: {result_text}")
plt_mod.show()